In [2]:
import random
import os
import json

db_path = '../data/images/ilhan-org'

names_to_id = []

def generate_id(name):
    return str(random.randint(10000, 99999))

for file in os.listdir(db_path):
    # Generate 5 digit random number for id, do it until it's unique
    name = file.split('.')[0]
    while True:
        id = generate_id(name)
        if id not in names_to_id:
            break
    
    names_to_id.append({
        'name': name,
        'id': id
    })

# Save names_to_id to a json file
with open('names_to_id.json', 'w') as f:
    json.dump(names_to_id, f)

In [3]:
import json
from datetime import datetime

conn = psycopg2.connect(
    dbname="smart-office",
    user="postgres",
    password="abuboy266#",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()

# First check if the deleted column exists
cursor.execute("SELECT column_name FROM information_schema.columns WHERE table_name='dates' AND column_name='deleted';")
if not cursor.fetchone():
    print("❌ 'deleted' column not found. Please run the database-fix-script.py first.")
    exit()

try:
    user_id = int(input("ID kiriting: ").strip())
except ValueError:
    print("❌ Noto'g'ri ID! Butun son bo'lishi kerak.")
    exit()

cursor.execute("SELECT username FROM users WHERE id = %s", (user_id,))
result = cursor.fetchone()

if not result:
    print("❌ Bunday IDga ega foydalanuvchi topilmadi.")
    exit()

username = result[0]

status = input("Status kiriting (in / out): ").strip().lower()

if status not in ["in", "out"]:
    print("❌ Noto'g'ri status! Faqat 'in' yoki 'out' bo'lishi mumkin.")
    exit()

now = datetime.now()

if status == "in":
    cursor.execute("""
        INSERT INTO dates (id, username, userIn, clientStatus, deleted)
        VALUES (%s, %s, %s, %s, FALSE)
    """, (user_id, username, now, status))
elif status == "out":
    cursor.execute("""
        INSERT INTO dates (id, username, userOut, clientStatus, deleted)
        VALUES (%s, %s, %s, %s, FALSE)
    """, (user_id, username, now, status))

conn.commit()
cursor.close()
conn.close()

print(f"{username} (ID: {user_id}) uchun '{status}' status saqlandi.")

OperationalError: connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
